In [2]:
import os
import glob
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.models import load_model
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.layers import Input, Conv1D, Flatten, Dense, Permute
from tensorflow.keras.regularizers import l2

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from numpy.lib.stride_tricks import as_strided

I0000 00:00:1788832561.010681   40755 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


## Loading CSV

In [3]:
DATASET_DIR = "../dataset/phase1"
CMS_TAG = "cms4"
SEQ_LEN = 165
DIVISOR = 200000.0
NUM_CLASSES = 2
CLASSES = np.array(['benign', 'malware'])

def load_data():
    bdir = os.path.join(DATASET_DIR, f'benign_{CMS_TAG}')
    mdir = os.path.join(DATASET_DIR, f'malware_{CMS_TAG}')
    bfiles = sorted(glob.glob(os.path.join(bdir, 'cms_*.csv')))
    mfiles = sorted(glob.glob(os.path.join(mdir, 'cms_*.csv')))

    def load_all(files):
        out = []
        for f in files:
            v = pd.read_csv(f, header=None).values.flatten().astype(np.float32)
            out.append(v)
        return np.array(out, dtype=np.float32)

    Xb = load_all(bfiles)
    Xm = load_all(mfiles)
    X = np.concatenate([Xb, Xm], axis=0)
    y = np.concatenate([np.zeros(len(Xb), dtype=np.int64), np.ones(len(Xm), dtype=np.int64)])
    return X, y

In [4]:
X, y = load_data()

## Train, Validation, Test Split and Normalize

In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

X_train = np.clip(X_train / DIVISOR, 0.0, 1.0)
X_val = np.clip(X_val / DIVISOR, 0.0, 1.0)
X_test = np.clip(X_test / DIVISOR, 0.0, 1.0)

X_train = X_train.reshape(-1, SEQ_LEN, 1)
X_val = X_val.reshape(-1, SEQ_LEN, 1)
X_test = X_test.reshape(-1, SEQ_LEN, 1)

## 1D CNN model

In [6]:
input_layer = Input(shape=(SEQ_LEN, 1))

x = Conv1D(filters=16, kernel_size=3, strides=10, padding='valid', activation='relu')(input_layer)
x = Conv1D(filters=32, kernel_size=3, strides=1, padding='valid', activation='relu')(x)
x = Conv1D(filters=64, kernel_size=3, strides=1, padding='valid', activation='relu')(x)

x = Permute((2, 1))(x)
x = Flatten()(x)

x = Dense(32, activation='relu', kernel_regularizer=l2(1e-4))(x)
output_layer = Dense(NUM_CLASSES, activation='softmax', kernel_regularizer=l2(1e-4))(x)

model = Model(input_layer, output_layer)

opt = Adam(learning_rate=0.001)
model.compile(loss='sparse_categorical_crossentropy', optimizer=opt, metrics=['accuracy'])

## Check Point

In [7]:
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=8, min_lr=0.00001)
checkpoint = ModelCheckpoint(
    filepath='./phase1_cms4.h5',
    save_best_only=True,
    monitor='val_accuracy',
    mode='max',
    verbose=1
)

## Model Training

In [8]:
model.fit(X_train, y_train, epochs=200, batch_size=32, validation_data=(X_val, y_val), callbacks=[reduce_lr, checkpoint])

Epoch 1/200
60/71 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7693 - loss: 0.5720
Epoch 1: val_accuracy improved from None to 0.84725, saving model to ./phase1_cms4.h5



Epoch 1: finished saving model to ./phase1_cms4.h5
71/71 ━━━━━━━━━━━━━━━━━━━━ 4s 13ms/step - accuracy: 0.7748 - loss: 0.5515 - val_accuracy: 0.8472 - val_loss: 0.3844 - learning_rate: 0.0010
Epoch 2/200
70/71 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8562 - loss: 0.3218
Epoch 2: val_accuracy improved from 0.84725 to 0.86323, saving model to ./phase1_cms4.h5



Epoch 2: finished saving model to ./phase1_cms4.h5
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.8565 - loss: 0.3220 - val_accuracy: 0.8632 - val_loss: 0.2595 - learning_rate: 0.0010
Epoch 3/200
61/71 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9129 - loss: 0.2212
Epoch 3: val_accuracy improved from 0.86323 to 0.92895, saving model to ./phase1_cms4.h5



Epoch 3: finished saving model to ./phase1_cms4.h5
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9138 - loss: 0.2183 - val_accuracy: 0.9290 - val_loss: 0.1907 - learning_rate: 0.0010
Epoch 4/200
63/71 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9286 - loss: 0.2004
Epoch 4: val_accuracy improved from 0.92895 to 0.93961, saving model to ./phase1_cms4.h5



Epoch 4: finished saving model to ./phase1_cms4.h5
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9316 - loss: 0.1930 - val_accuracy: 0.9396 - val_loss: 0.1993 - learning_rate: 0.0010
Epoch 5/200
59/71 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9354 - loss: 0.1777
Epoch 5: val_accuracy improved from 0.93961 to 0.94494, saving model to ./phase1_cms4.h5


...


Epoch 30: finished saving model to ./phase1_cms4.h5
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.9618 - loss: 0.1223 - val_accuracy: 0.9627 - val_loss: 0.1179 - learning_rate: 0.0010
Epoch 31/200
69/71 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9597 - loss: 0.1200
Epoch 31: val_accuracy did not improve from 0.96270
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.9596 - loss: 0.1203 - val_accuracy: 0.9574 - val_loss: 0.1209 - learning_rate: 0.0010
Epoch 32/200
70/71 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9589 - loss: 0.1205
Epoch 32: val_accuracy did not improve from 0.96270
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.9591 - loss: 0.1202 - val_accuracy: 0.9627 - val_loss: 0.1177 - learning_rate: 0.0010
Epoch 33/200
67/71 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9594 - loss: 0.1251
Epoch 33: val_accuracy improved from 0.96270 to 0.96448, saving model to ./phase1_cms4.h5



Epoch 33: finished saving model to ./phase1_cms4.h5
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.9596 - loss: 0.1249 - val_accuracy: 0.9645 - val_loss: 0.1247 - learning_rate: 0.0010
Epoch 34/200
63/71 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9593 - loss: 0.1223
Epoch 34: val_accuracy did not improve from 0.96448
71/71 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9582 - loss: 0.1261 - val_accuracy: 0.9609 - val_loss: 0.1206 - learning_rate: 0.0010
Epoch 35/200
63/71 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9628 - loss: 0.1162
Epoch 35: val_accuracy did not improve from 0.96448
71/71 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9618 - loss: 0.1188 - val_accuracy: 0.9574 - val_loss: 0.1185 - learning_rate: 0.0010
Epoch 36/200
63/71 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9559 - loss: 0.1231
Epoch 36: val_accuracy did not improve from 0.96448
71/71 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9587 - loss: 0.1165 - val_accuracy: 0.9538 - val_loss: 0.1352 


Epoch 39: finished saving model to ./phase1_cms4.h5
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9618 - loss: 0.1173 - val_accuracy: 0.9663 - val_loss: 0.1124 - learning_rate: 0.0010
Epoch 40/200
59/71 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9629 - loss: 0.1140
Epoch 40: val_accuracy did not improve from 0.96625
71/71 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9622 - loss: 0.1164 - val_accuracy: 0.9609 - val_loss: 0.1172 - learning_rate: 0.0010
Epoch 41/200
64/71 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9629 - loss: 0.1174
Epoch 41: val_accuracy did not improve from 0.96625
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9649 - loss: 0.1114 - val_accuracy: 0.9591 - val_loss: 0.1140 - learning_rate: 0.0010
Epoch 42/200
68/71 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9628 - loss: 0.1123
Epoch 42: val_accuracy did not improve from 0.96625
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9622 - loss: 0.1135 - val_accuracy: 0.9609 - val_loss: 0.1127 -


Epoch 44: finished saving model to ./phase1_cms4.h5
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9627 - loss: 0.1153 - val_accuracy: 0.9680 - val_loss: 0.1099 - learning_rate: 0.0010
Epoch 45/200
60/71 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9630 - loss: 0.1150
Epoch 45: val_accuracy did not improve from 0.96803
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9627 - loss: 0.1154 - val_accuracy: 0.9449 - val_loss: 0.1292 - learning_rate: 0.0010
Epoch 46/200
59/71 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9666 - loss: 0.1059
Epoch 46: val_accuracy did not improve from 0.96803
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9645 - loss: 0.1133 - val_accuracy: 0.9520 - val_loss: 0.1295 - learning_rate: 0.0010
Epoch 47/200
66/71 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9654 - loss: 0.1141
Epoch 47: val_accuracy did not improve from 0.96803
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9653 - loss: 0.1126 - val_accuracy: 0.9627 - val_loss: 0.1118 -


Epoch 49: finished saving model to ./phase1_cms4.h5
71/71 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9640 - loss: 0.1122 - val_accuracy: 0.9698 - val_loss: 0.1104 - learning_rate: 0.0010
Epoch 50/200
69/71 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9638 - loss: 0.1130
Epoch 50: val_accuracy did not improve from 0.96980
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9645 - loss: 0.1117 - val_accuracy: 0.9485 - val_loss: 0.1165 - learning_rate: 0.0010
Epoch 51/200
65/71 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9668 - loss: 0.1046
Epoch 51: val_accuracy did not improve from 0.96980
71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9640 - loss: 0.1110 - val_accuracy: 0.9467 - val_loss: 0.1708 - learning_rate: 0.0010
Epoch 52/200
69/71 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9615 - loss: 0.1212
Epoch 52: val_accuracy did not improve from 0.96980
71/71 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9614 - loss: 0.1218 - val_accuracy: 0.9680 - val_loss: 0.1112 -

## Evaluate (float32)

In [9]:
cp_model = load_model('./phase1_cms4.h5')
cp_model.evaluate(X_test, y_test, batch_size=1000)

2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step - accuracy: 0.9652 - loss: 0.1124 


[0.11243235319852829, 0.96517413854599]

In [10]:
y_pred = cp_model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)

print(classification_report(y_test, y_pred_classes, target_names=list(CLASSES), digits=4))

38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step
              precision    recall  f1-score   support

      benign     0.9476    0.9872    0.9670       623
     malware     0.9856    0.9417    0.9632       583

    accuracy                         0.9652      1206
   macro avg     0.9666    0.9644    0.9651      1206
weighted avg     0.9660    0.9652    0.9651      1206



In [11]:
conf_matrix = confusion_matrix(y_test, y_pred_classes)
conf_matrix_df = pd.DataFrame(conf_matrix, index=list(CLASSES), columns=list(CLASSES))
print("Confusion Matrix:")
print(conf_matrix_df)

Confusion Matrix:
         benign  malware
benign      615        8
malware      34      549


## Q15 Quantization

In [12]:
def scale_for_q15(w):
    absmax = np.abs(w).max()
    if absmax <= 1.0:
        return w, 1
    scale = 1
    while absmax / scale > 1.0:
        scale *= 2
    return w / scale, scale

def to_q15(x):
    return np.clip(np.round(x * 32768.0), -32768, 32767).astype(np.int64)

def to_q15_bias(x):
    return np.clip(np.round(x * 32768.0), -2**31, 2**31 - 1).astype(np.int64)

def quantize_weights_flatten(model):
    conv_layers = [l for l in model.layers if 'conv1d' in l.name]
    dense_layers = [l for l in model.layers if l.name.startswith('dense')]
    c1_w_f, c1_b_f = conv_layers[0].get_weights()
    c2_w_f, c2_b_f = conv_layers[1].get_weights()
    c3_w_f, c3_b_f = conv_layers[2].get_weights()
    d_w_f, d_b_f = dense_layers[0].get_weights()
    fc_w_f, fc_b_f = dense_layers[1].get_weights()
    c1_w = np.transpose(c1_w_f[:, 0, :], (1, 0))
    c2_w = np.transpose(c2_w_f, (2, 1, 0))
    c3_w = np.transpose(c3_w_f, (2, 1, 0))
    d_w = d_w_f.T
    fc_w = fc_w_f.T
    out = {}
    for name, w, b in [('conv1', c1_w, c1_b_f), ('conv2', c2_w, c2_b_f), ('conv3', c3_w, c3_b_f),
                        ('dense', d_w, d_b_f), ('fc2', fc_w, fc_b_f)]:
        w_s, scale = scale_for_q15(w)
        out[f'{name}_w'] = to_q15(w_s)
        out[f'{name}_b'] = to_q15_bias(b / scale)
    return out

def windows_1d(a, out_len, k, stride, axis):
    a = np.ascontiguousarray(a)
    shape = list(a.shape); shape[axis] = out_len; shape = shape + [k]
    strides = list(a.strides); ts = strides[axis]
    strides[axis] = ts * stride; strides = strides + [ts]
    return as_strided(a, shape=shape, strides=strides)

def q15_forward(qw, X, s1, s2, s3, seq_len):
    c1w, c1b = qw['conv1_w'].astype(np.int64), qw['conv1_b'].astype(np.int64)
    c2w, c2b = qw['conv2_w'].astype(np.int64), qw['conv2_b'].astype(np.int64)
    c3w, c3b = qw['conv3_w'].astype(np.int64), qw['conv3_b'].astype(np.int64)
    dw, db = qw['dense_w'].astype(np.int64), qw['dense_b'].astype(np.int64)
    fcw, fcb = qw['fc2_w'].astype(np.int64), qw['fc2_b'].astype(np.int64)
    F1, K1 = c1w.shape
    F2, _, K2 = c2w.shape
    F3, _, K3 = c3w.shape
    c1out = (seq_len - K1) // s1 + 1
    c2out = (c1out - K2) // s2 + 1
    c3out = (c2out - K3) // s3 + 1
    Xq = np.clip(np.round(X * 32768), -32768, 32767).astype(np.int64)
    win1 = windows_1d(Xq, c1out, K1, s1, axis=1)
    a1 = np.maximum(0, (np.einsum('ntk,fk->nft', win1, c1w) >> 15) + c1b[None, :, None])
    a1c = np.clip(a1, -32768, 32767)
    win2 = windows_1d(a1c, c2out, K2, s2, axis=2)
    a2 = np.maximum(0, (np.einsum('nctk,fck->nft', win2, c2w) >> 15) + c2b[None, :, None])
    a2c = np.clip(a2, -32768, 32767)
    win3 = windows_1d(a2c, c3out, K3, s3, axis=2)
    a3 = np.maximum(0, (np.einsum('nctk,fck->nft', win3, c3w) >> 15) + c3b[None, :, None])
    a3c = np.clip(a3, -32768, 32767)
    N = X.shape[0]
    flat = a3c.reshape(N, F3 * c3out)
    hidden = np.clip(np.maximum(0, (flat @ dw.T >> 15) + db[None, :]), -32768, 32767)
    logits = (hidden @ fcw.T >> 15) + fcb[None, :]
    return np.argmax(logits, axis=1)

## Evaluate (Q15)

In [13]:
qw = quantize_weights_flatten(cp_model)
X_test_flat = X_test.reshape(-1, SEQ_LEN)
y_pred_q15 = q15_forward(qw, X_test_flat, 10, 1, 1, SEQ_LEN)

print(classification_report(y_test, y_pred_q15, target_names=list(CLASSES), digits=4))

              precision    recall  f1-score   support

      benign     0.9472    0.9502    0.9487       623
     malware     0.9466    0.9434    0.9450       583

    accuracy                         0.9469      1206
   macro avg     0.9469    0.9468    0.9469      1206
weighted avg     0.9469    0.9469    0.9469      1206



In [14]:
conf_matrix_q15 = confusion_matrix(y_test, y_pred_q15)
conf_matrix_q15_df = pd.DataFrame(conf_matrix_q15, index=list(CLASSES), columns=list(CLASSES))
print("Confusion Matrix (Q15):")
print(conf_matrix_q15_df)

Confusion Matrix (Q15):
         benign  malware
benign      592       31
malware      33      550
